<a href="https://colab.research.google.com/github/AndrewHung87/startup-funding-analysis/blob/main/startup_funding_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Startup Funding Analysis
**Data Source:** Crunchbase Startup Dataset (1,827 records)  
**Objective:** Clean and explore startup data to uncover funding trends, industry distribution, and growth patterns.

**Tools:** Python, Pandas, Matplotlib, Seaborn

In [ ]:
import pandas as pd

# Load dataset
url = 'https://raw.githubusercontent.com/AndrewHung87/startup-funding-analysis/refs/heads/main/Startup_Data__Startup.csv'
df = pd.read_csv(url)
# Basic overview
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)

Shape: (1827, 34)

Data Types:
Organization_Name                    object
Organization_Name_URL                object
Founded_Date                         object
Founders                             object
Number_of_Employees                  object
Description                          object
Industries                           object
Industry_Groups                      object
Headquarters_Location                object
Headquarters_Regions                 object
Postal_Code                          object
Diversity_Spotlight                  object
Estimated_Revenue_Range              object
Operating_Status                     object
Funding_Status                       object
Number_of_Funding_Rounds              int64
Last_Funding_Amount                  object
Last_Funding_Amount_Currency         object
Last_Funding_Date                    object
Last_Funding_Type                    object
Total_Funding_Amount                 object
Total_Funding_Amount_Currency        object
N

## Step 2: Data Cleaning

### 2.1 Missing Values Analysis
Before cleaning, we first identify which columns have missing values and how severe they are.

In [ ]:
# Calculate missing values count and percentage
missing_count = df.isnull().sum()
missing_pct = df.isnull().sum() / len(df) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct.round(2)
}).sort_values('Missing %', ascending=False)

# Show only columns that have missing values
missing_df[missing_df['Missing Count'] > 0]

,Missing Count,Missing %
Last_Layoff_Mention_Date,1823,99.78
Diversity_Spotlight,1735,94.96
Last_Leadership_Hiring_Date,1641,89.82
Estimated_Revenue_Range,1546,84.62
Additional_Comment,1529,83.69
Job_postings_on_official_website,1153,63.11
Postal_Code,772,42.26
Number_of_Lead_Investors,663,36.29
Founders,561,30.71
Top_5_Investors,407,22.28


### 2.2 Dropping Columns
Remove columns that are too sparse or unreliable to be useful in analysis.

In [ ]:
# Columns to drop: too many missing values, unreliable data, or redundant
cols_to_drop = [
    'Last_Layoff_Mention_Date',       # 99.78% missing
    'Diversity_Spotlight',             # 94.96% missing
    'Last_Leadership_Hiring_Date',     # 89.82% missing
    'Additional_Comment',              # 83.69% missing, unstructured free text
    'Estimated_Revenue_Range',         # 84.62% missing
    'Postal_Code',                     # 42.26% missing, using Headquarters_Regions instead
    'Job_Postings_on_Linkedin',        # unreliable data collection method
    'Job_postings_on_official_website' # unreliable data collection method
]

df.drop(columns=cols_to_drop, inplace=True)

print(f"Remaining columns: {df.shape[1]}")
print(df.columns.tolist())

Remaining columns: 26
['Organization_Name', 'Organization_Name_URL', 'Founded_Date', 'Founders', 'Number_of_Employees', 'Description', 'Industries', 'Industry_Groups', 'Headquarters_Location', 'Headquarters_Regions', 'Operating_Status', 'Funding_Status', 'Number_of_Funding_Rounds', 'Last_Funding_Amount', 'Last_Funding_Amount_Currency', 'Last_Funding_Date', 'Last_Funding_Type', 'Total_Funding_Amount', 'Total_Funding_Amount_Currency', 'Number_of_Lead_Investors', 'Top_5_Investors', 'Trend_Score_7_Days', 'Trend_Score_30_Days', 'Trend_Score_90_Days', 'Growth_Trend', 'Active_Hiring']


### 2.3 Filling Missing Values
Apply targeted imputation strategies based on each column's context.

In [ ]:
# --- Funding Amounts ---
# Fill Total_Funding_Amount with Last_Funding_Amount as a conservative estimate
df['Total_Funding_Amount'] = df['Total_Funding_Amount'].fillna(df['Last_Funding_Amount'])

# Fill remaining Last_Funding_Amount with 0 (no recent funding recorded)
df['Last_Funding_Amount'] = df['Last_Funding_Amount'].fillna(0)
df['Total_Funding_Amount'] = df['Total_Funding_Amount'].fillna(0)

# --- Currency ---
# Assume USD for missing currency fields (dataset is primarily US-based)
df['Last_Funding_Amount_Currency'] = df['Last_Funding_Amount_Currency'].fillna('USD')
df['Total_Funding_Amount_Currency'] = df['Total_Funding_Amount_Currency'].fillna('USD')

# --- Categorical Columns ---
# Fill with 'Unknown' to preserve row and flag missing data explicitly
cols_fill_unknown = [
    'Industries',
    'Industry_Groups',
    'Growth_Trend',
    'Funding_Status',
    'Founders',
    'Top_5_Investors'
]
df[cols_fill_unknown] = df[cols_fill_unknown].fillna('Unknown')

# --- Numeric Columns ---
# Fill with 0 — absence of record implies no leads/investors found
df['Number_of_Lead_Investors'] = df['Number_of_Lead_Investors'].fillna(0)

# --- Verify no missing values remain ---
remaining_missing = df.isnull().sum()
print("Remaining missing values:")
print(remaining_missing[remaining_missing > 0])
print("\nIf nothing printed above, all missing values are handled!")

Remaining missing values:
Series([], dtype: int64)

If nothing printed above, all missing values are handled!


### 2.4 Fixing Data Types
Convert columns to their correct data types:
- Date columns: `object` → `datetime`
- Funding amount columns: `object` → `numeric`

In [ ]:
# --- Date Columns ---
date_cols = [
    'Founded_Date',
    'Last_Funding_Date',
    'Last_Layoff_Mention_Date' if 'Last_Layoff_Mention_Date' in df.columns else None
]
date_cols = [col for col in date_cols if col is not None and col in df.columns]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# --- Funding Amount Columns ---
# Remove commas and convert to numeric
df['Last_Funding_Amount'] = pd.to_numeric(
    df['Last_Funding_Amount'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
).fillna(0)

df['Total_Funding_Amount'] = pd.to_numeric(
    df['Total_Funding_Amount'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
).fillna(0)

# --- Verify ---
print("Updated Data Types:")
print(df[['Founded_Date', 'Last_Funding_Date', 'Last_Funding_Amount', 'Total_Funding_Amount']].dtypes)

print("\nSample values:")
print(df[['Founded_Date', 'Last_Funding_Date', 'Last_Funding_Amount', 'Total_Funding_Amount']].head())

Updated Data Types:
Founded_Date            datetime64[ns]
Last_Funding_Date       datetime64[ns]
Last_Funding_Amount            float64
Total_Funding_Amount           float64
dtype: object

Sample values:
  Founded_Date Last_Funding_Date  Last_Funding_Amount  Total_Funding_Amount
0   2023-01-01        2024-02-29                  0.0                   0.0
1   2024-01-01        2025-01-01                  0.0                   0.0
2   2024-01-01        2024-09-13                  0.0                   0.0
3   2023-08-01        2024-11-20                  0.0                   0.0
4   2023-01-01        2024-11-13                  0.0                   0.0
